In [ ]:
#@title ⚙️ Setup — choose how to provide the inputs (RUN ME FIRST)
#@markdown Pick how the input files for this step are provided:
#@markdown - **Mobile mode**: the files are downloaded automatically from the GitHub repo — no manual upload, works on phones/tablets.
#@markdown - **Manual upload**: you upload the files yourself in the cells below.
input_mode = "Mobile mode (auto-download from repo)" #@param ["Mobile mode (auto-download from repo)", "Manual upload"]

import os, urllib.request
REPO_RAW = "https://raw.githubusercontent.com/biochorl/Nanobody_de_novo_design/main"

# Inputs needed by THIS step.  (some links are FACSIMILE placeholders until the files
# are added to the repo — they will start working once they are committed)
INPUT_FILES = [('/content/PIPPack_outputs.zip', '/Example_output/PIPPack_outputs.zip', 'facsimile'), ('/content/IgGM_input.fasta', '/Example_output/IgGM_input.fasta', 'facsimile')]

def fetch_inputs(files):
    ok = 0
    for dst, rel, kind in files:
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        url = REPO_RAW + rel
        try:
            urllib.request.urlretrieve(url, dst)
            print(f"✅ {os.path.basename(dst):<28} ← {url}  [{kind}]")
            ok += 1
        except Exception as e:
            tag = "facsimile not in repo yet" if kind == "facsimile" else "ERROR"
            print(f"⚠️  {os.path.basename(dst):<28} could not be downloaded ({tag}): {e}")
    return ok

if input_mode.startswith("Mobile"):
    print("📱 Mobile mode — downloading this step's inputs from the repo:\n")
    n = fetch_inputs(INPUT_FILES)
    print(f"\n→ {n}/{len(INPUT_FILES)} file(s) ready in /content. "
          "Skip the manual-upload cells and point the path fields to these files.")
else:
    print("🖱️ Manual mode — upload your files in the cells below as usual.")


# **AntiFold for nanobody CDRs (re)design**
## Overview
####[Antifold](https://github.com/oxpig/AntiFold) ([paper link](https://academic.oup.com/bioinformaticsadvances/article/5/1/vbae202/8090019)) is a model trained to reconstruct the sequence of complementarity-determining regions (CDRs) of antibody- or nanobody-antigen complexes. It has a superior accuracy on CDRs with respect to ProteinMPNN
---

In [ ]:
# @title Install AntiFold and Dependencies
# @markdown Run this cell first to install all required packages

# Install Miniconda
!wget https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -q
!chmod +x Miniconda3-latest-Linux-x86_64.sh
!bash ./Miniconda3-latest-Linux-x86_64.sh -b -f -p /usr/local

# Accept Conda Terms of Service using the full path
!/usr/local/bin/conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
!/usr/local/bin/conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r

# Initialize conda
!source /usr/local/bin/activate
!/usr/local/bin/conda init bash

# Set up conda environment
!/usr/local/bin/conda create --name antifold python=3.10 -y

# Use the full path to the conda environment to avoid activation issues
!/usr/local/envs/antifold/bin/pip install torch==2.2.0 --index-url https://download.pytorch.org/whl/cpu

# Clone and install AntiFold
!git clone https://github.com/oxpig/AntiFold
%cd AntiFold
!/usr/local/envs/antifold/bin/pip install .

# Install additional dependencies for the notebook
!/usr/local/envs/antifold/bin/pip install biopython pandas ipywidgets

# Create necessary directories
!mkdir -p data/pdbs
!mkdir -p output

print("Installation completed successfully!")

## Upload Files and Run AntiFold
Upload the ZIP containing your PDB designs (e.g. from PIPPack) and the `nanobody_redesign.fasta` generated in step 1. AntiFold will automatically detect the `X` residues and mutate strictly those positions.

In [ ]:
# @title Run AntiFold Redesign
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import os
import zipfile
import subprocess
import glob
import pandas as pd
from google.colab import files

# --- UI Form ---
display(HTML("<h2>Nanobody Design Parameters</h2>"))
num_sequences = widgets.IntSlider(description="Number of sequences:", min=1, max=100, value=10)
seed_value = widgets.Text(description="Random seed:", value="5748")
temperature = widgets.FloatSlider(description="Sampling temperature:", min=0.1, max=1.0, step=0.1, value=0.3)
use_json_annotation = widgets.Checkbox(value=True, description='Use dynamic CDR JSON (recommended)')
display(num_sequences, seed_value, temperature, use_json_annotation)

display(HTML("<h3>1. Upload nanobody_redesign.fasta (or IgGM_input.fasta)</h3>"))
upload_fasta = widgets.FileUpload(description="Choose FASTA", accept='.fasta', multiple=False)
display(upload_fasta)

display(HTML("<h3>2. Upload PIPPack_outputs.zip (or PDBs)</h3>"))
upload_pdbs = widgets.FileUpload(description="Choose ZIP/PDB", accept='.zip,.pdb', multiple=True)
display(upload_pdbs)

run_button = widgets.Button(description="Run AntiFold", button_style='success')
output_area = widgets.Output()

def get_fasta_mask(fasta_path):
    # Parse the nanobody fasta to find X
    seq = ""
    with open(fasta_path, 'r') as f:
        lines = f.readlines()
        for line in lines:
            if line.startswith(">H") or line.startswith(">original_nanobody"):
                continue
            if line.startswith(">"):
                break # Reached antigen
            seq += line.strip()

    x_indices = [i for i, aa in enumerate(seq) if aa.upper() == 'X']
    return x_indices

def on_run_button_clicked(b):
    with output_area:
        clear_output(wait=True)
        print("Starting process...")

        if not upload_fasta.value or not upload_pdbs.value:
            print("❌ Please upload both the FASTA file and the PIPPack ZIP/PDBs.")
            return

        # 1. Save FASTA
        fasta_info = list(upload_fasta.value.values())[0]
        fasta_name = list(upload_fasta.value.keys())[0]
        with open(fasta_name, 'wb') as f:
            f.write(fasta_info['content'])
        print(f"✅ FASTA '{fasta_name}' uploaded.")

        x_indices = get_fasta_mask(fasta_name)
        print(f"✅ Found {len(x_indices)} 'X' residues to redesign at indices: {x_indices}")

        # 2. Extract PDBs
        input_dir = "data/pdbs"
        os.makedirs(input_dir, exist_ok=True)
        # clean dir
        for f in glob.glob(f"{input_dir}/*"):
            try:
                os.remove(f)
            except:
                pass

        for fn, finfo in upload_pdbs.value.items():
            path = os.path.join(input_dir, fn)
            with open(path, 'wb') as f:
                f.write(finfo['content'])
            if fn.endswith('.zip'):
                with zipfile.ZipFile(path, 'r') as zip_ref:
                    for member in zip_ref.namelist():
                        basename = os.path.basename(member)
                        # Skip directory entries, macOS metadata, and hidden files
                        if not basename or basename.startswith("._") or "__MACOSX" in member:
                            continue
                        # Flatten files into input_dir, only keeping PDBs
                        if basename.endswith(".pdb"):
                            if basename.startswith("temp_") or basename.startswith("refined_"):
                                continue
                            content = zip_ref.read(member)
                            with open(os.path.join(input_dir, basename), 'wb') as out_f:
                                out_f.write(content)
                os.remove(path)

        pdbs = glob.glob(f"{input_dir}/*.pdb")
        print(f"✅ Extracted {len(pdbs)} PDBs.")

        if len(pdbs) == 0:
            print("❌ No valid PDBs found in the uploaded zip.")
            return

        # 3. Write Custom Python Runner using replacement instead of f-string
        runner_code = r"""import sys
import os
import glob
import pandas as pd
import numpy as np
import Bio.PDB
import zipfile
import json

# Add antifold to path
sys.path.append(os.path.abspath("."))
import antifold.main
import antifold.antiscripts

# --- DYNAMIC CDR MASKING WITH COMPLEX SUPPORT ---
orig_get_imgt_mask = antifold.antiscripts.get_imgt_mask

def custom_imgt_mask(df, imgt_regions):
    if "CUSTOM_FASTA" in imgt_regions:
        mask = np.zeros(len(df), dtype=bool)

        use_json = __USE_JSON__
        annot_file = os.path.join("__INPUT_DIR__", "cdrs_annotation.json")

        length_to_mask = {}
        if use_json and os.path.exists(annot_file):
            with open(annot_file, "r") as f:
                annotations = json.load(f)
            for pdb_name, bounds in annotations.items():
                indices = []
                for cdr, (start, end) in bounds.items():
                    indices.extend(range(start - 1, end)) # 0-based
                # length of nanobody = CDR3_end + tail_length (which is 10)
                length = bounds["CDR3"][1] + 10
                length_to_mask[length] = indices

        # Find the nanobody length in df to pick the correct mask
        h_len = sum(1 for i in range(len(df)) if df.iloc[i].get('chain', 'H') == 'H')

        if use_json and h_len in length_to_mask:
            mask_indices = length_to_mask[h_len]
            print(f"Applied dynamic JSON mask to {len(mask_indices)} residues (for nanobody length {h_len}).")
        else:
            mask_indices = __MASK_INDICES__
            print(f"Applied fallback FASTA mask to {len(mask_indices)} residues.")

        # Apply mask ONLY to the nanobody residues
        nb_idx = 0
        for i in range(len(df)):
            if df.iloc[i].get('chain', 'H') == 'H':
                if nb_idx in mask_indices:
                    mask[i] = True
                nb_idx += 1

        return mask
    return orig_get_imgt_mask(df, imgt_regions)

antifold.antiscripts.get_imgt_mask = custom_imgt_mask
# ------------------------------------------------

# We process the FULL complex (from input_dir)
src_pdbs = glob.glob("__INPUT_DIR__/*.pdb")

parser = Bio.PDB.PDBParser(QUIET=True)
valid_pdbs = []
valid_rows = []

for p in src_pdbs:
    basename = os.path.basename(p)
    if basename.startswith("._") or basename.startswith("temp_") or basename.startswith("refined_"):
        continue
    try:
        pdb_id = os.path.splitext(basename)[0]
        structure = parser.get_structure('pdb', p)
        if len(structure) == 0: continue
        chains = list(structure[0].get_chains())
        if len(chains) == 0: continue

        chain_ids = [ch.id for ch in chains]
        nb_chain_id = 'H' if 'H' in chain_ids else min(structure[0], key=lambda c: len(list(c.get_residues()))).id

        # Identify antigen chains
        ag_chains = [ch.id for ch in chains if ch.id != nb_chain_id]
        ag_chain_id = ag_chains[0] if ag_chains else ""

        valid_rows.append({"pdb": pdb_id, "Hchain": nb_chain_id, "antigen_chain": ag_chain_id})
        valid_pdbs.append(p)
        print(f"{pdb_id}: Nanobody '{nb_chain_id}', Antigen '{ag_chain_id}'")

    except Exception as e:
        print(f"Warning: Failed to parse PDB {p}: {e}")
        continue

pdbs_csv = pd.DataFrame(valid_rows, columns=["pdb", "Hchain", "antigen_chain"])

if pdbs_csv.empty:
    print("❌ Error: No valid design PDBs found for AntiFold to run on.")
    sys.exit(1)

print(f"Running AntiFold on {len(valid_pdbs)} full complexes...")
model = antifold.antiscripts.load_model()

res = antifold.main.sample_pdbs(
    model=model,
    pdbs_csv_or_dataframe=pdbs_csv,
    out_dir="output",
    pdb_dir="__INPUT_DIR__",
    regions_to_mutate=["CUSTOM_FASTA"],
    sample_n=__SAMPLE_N__,
    sampling_temp=__SAMPLING_TEMP__,
    seed=__SEED__,
    save_flag=True,
    nanobody_mode=True,
    custom_chain_mode=True
)

print("\n--- Filtering Top 1 Sequences ---")
out_zip = "AntiFold_Best_Designs.zip"
with zipfile.ZipFile(out_zip, 'w') as zf:
    with open("top1_sequences.fasta", "w") as fa:
        for pdb_id, out_dict in res.items():
            best_seq = ""
            best_score = -float('inf')

            for seq_id, seq_record in out_dict['sequences'].items():
                if "__" not in seq_id: continue
                desc = seq_record.description
                score_str = [x for x in desc.split(',') if 'score=' in x and 'global_score' not in x]
                if score_str:
                    score = float(score_str[0].split('=')[1])
                    if score > best_score:
                        best_score = score
                        best_seq = str(seq_record.seq)
            fa.write(f'>{pdb_id} score={best_score:.3f}\n{best_seq}\n')

    zf.write("top1_sequences.fasta")
    for p in valid_pdbs:
        zf.write(p, os.path.basename(p))

print("AntiFold finished.")"""
        runner_code = runner_code.replace("__MASK_INDICES__", str(x_indices))
        runner_code = runner_code.replace("__SAMPLE_N__", str(num_sequences.value))
        runner_code = runner_code.replace("__SAMPLING_TEMP__", str(temperature.value))
        runner_code = runner_code.replace("__SEED__", str(seed_value.value))
        runner_code = runner_code.replace("__USE_JSON__", "True" if use_json_annotation.value else "False")
        runner_code = runner_code.replace("__INPUT_DIR__", input_dir)

        with open("run_antifold_custom.py", "w") as f:
            f.write(runner_code)

        print("\n🚀 Running AntiFold... this may take a few minutes.")
        try:
            result = subprocess.run(["/usr/local/envs/antifold/bin/python", "run_antifold_custom.py"], capture_output=True, text=True, check=True)
            print("\n--- AntiFold Output ---")
            print(result.stdout)
            print("✅ AntiFold completed successfully!")

            out_zip = "AntiFold_Best_Designs.zip"
            print(f"✅ Created {out_zip} with top 1 sequences and original PDBs.")
            print("Downloading...")
            files.download(out_zip)

        except subprocess.CalledProcessError as e:
            print("\n❌ ERROR: AntiFold failed!")
            print(e.stdout)
            print(e.stderr)

run_button.on_click(on_run_button_clicked)
display(run_button, output_area)
